# 04_logistic_regression.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports & helpers

In [1]:
import pandas as pd
import numpy as np
import joblib, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (classification_report, f1_score,
                             precision_recall_curve, auc, roc_auc_score,
                             confusion_matrix)
from imblearn.over_sampling import SMOTE
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

COINS   = ["Bitcoin", "Ethereum", "Dogecoin"]
results = {}

def load_feature_cols(coin: str):
    path = f"feature_cols_{coin.lower()}.csv"
    return pd.read_csv(path)["feature"].tolist()


## Cell 2 — Utility: train/val/test split (time-series safe)

In [2]:
def time_split(df, val_frac=0.15, test_frac=0.15):
    """Chronological split — no shuffling."""
    n = len(df)
    train_end = int(n * (1 - val_frac - test_frac))
    val_end   = int(n * (1 - test_frac))
    return (df.iloc[:train_end].copy(),
            df.iloc[train_end:val_end].copy(),
            df.iloc[val_end:].copy())

## Cell 3 — Utility: print & collect metrics

In [3]:
def evaluate(name, coin, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    prec, rec, thresholds_pr = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(rec, prec)
    roc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    print(f"\n{'='*50}")
    print(f"{name} | {coin}")
    print(classification_report(y_true, y_pred, target_names=["Down","Up"]))
    print(f"ROC-AUC: {roc:.4f}   PR-AUC: {pr_auc:.4f}   F1(Up): {f1:.4f}")
    return {"model": name, "coin": coin,
            "f1_up": f1, "roc_auc": roc, "pr_auc": pr_auc,
            "threshold": threshold}

## Cell 4 — Train Logistic Regression per coin

In [4]:
lr_models = {}

for coin in COINS:
    feat_cols = load_feature_cols(coin)
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)
    train, val, test = time_split(df)

    X_train = train[feat_cols].values;  y_train = train["target"].values.astype(int)
    X_val   = val[feat_cols].values;    y_val   = val["target"].values.astype(int)
    X_test  = test[feat_cols].values;   y_test  = test["target"].values.astype(int)

    # SMOTE on train only
    smote = SMOTE(random_state=42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

    # Pipeline: StandardScaler -> Logistic Regression
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced"))
    ])

    # Hyperparameter tuning: C (regularisation strength)
    param_grid = {"lr__C": [0.01, 0.1, 1.0, 10.0]}
    tscv = TimeSeriesSplit(n_splits=4)
    gs = GridSearchCV(pipe, param_grid, cv=tscv, scoring="f1", n_jobs=-1, refit=True)
    gs.fit(X_train_sm, y_train_sm)

    best = gs.best_estimator_
    print(f"{coin} best C={gs.best_params_} | features={len(feat_cols)}")

    # Choose threshold on val set to maximise F1
    val_prob = best.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0.3, 0.7, 41)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val, (val_prob >= t).astype(int)))
    print(f"  Best threshold on val: {best_thr:.2f}")

    # Evaluate on test
    test_prob = best.predict_proba(X_test)[:, 1]
    res = evaluate("Logistic Regression", coin, y_test, test_prob, threshold=best_thr)
    results[f"lr_{coin}"] = res

    lr_models[coin] = {"model": best, "threshold": best_thr, "feature_cols": feat_cols}

print("\n✅ Logistic Regression complete for all coins.")


Bitcoin best C={'lr__C': 0.01} | features=60
  Best threshold on val: 0.30

Logistic Regression | Bitcoin
              precision    recall  f1-score   support

        Down       0.00      0.00      0.00       277
          Up       0.54      1.00      0.70       329

    accuracy                           0.54       606
   macro avg       0.27      0.50      0.35       606
weighted avg       0.29      0.54      0.38       606

ROC-AUC: 0.5288   PR-AUC: 0.5765   F1(Up): 0.7024
Ethereum best C={'lr__C': 0.01} | features=51
  Best threshold on val: 0.39

Logistic Regression | Ethereum
              precision    recall  f1-score   support

        Down       0.46      0.20      0.28       123
          Up       0.60      0.83      0.70       174

    accuracy                           0.57       297
   macro avg       0.53      0.52      0.49       297
weighted avg       0.54      0.57      0.52       297

ROC-AUC: 0.5045   PR-AUC: 0.5764   F1(Up): 0.6954
Dogecoin best C={'lr__C': 0.1} |

## Cell 5 — PR Curve plots

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, coin in zip(axes, COINS):
    feat_cols = lr_models[coin]["feature_cols"]
    df = pd.read_csv(f"features_{coin.lower()}.csv", parse_dates=["date"])
    df = df.sort_values("date")
    _, _, test = time_split(df)
    X_test = test[feat_cols].values
    y_test = test["target"].values.astype(int)
    probs = lr_models[coin]["model"].predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ax.plot(rec, prec, color="#3b82f6", linewidth=2)
    ax.fill_between(rec, prec, alpha=0.15, color="#3b82f6")
    ax.set_title(f"{coin}\nLR Precision-Recall Curve")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    baseline = y_test.mean()
    ax.axhline(baseline, color="gray", linestyle="--", label=f"Baseline {baseline:.2f}")
    ax.legend()
plt.suptitle("Logistic Regression — PR Curves", fontsize=13)
plt.tight_layout()
plt.savefig("lr_pr_curves.png", dpi=150)
plt.show()


## Cell 6 — Save LR models

In [6]:
for coin in COINS:
    joblib.dump(lr_models[coin]["model"], f"lr_model_{coin.lower()}.joblib")
print("✅ LR models saved.")

✅ LR models saved.
